In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("debajyotipodder/co2-emission-by-vehicles")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\bharani\.cache\kagglehub\datasets\debajyotipodder\co2-emission-by-vehicles\versions\1


In [2]:
!pip install joblib
!pip install scikit-learn

In [3]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
ds = pd.read_csv("CO2_Emissions.csv")

In [5]:
ds.head()

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
0,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244


In [6]:
ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7385 entries, 0 to 7384
Data columns (total 12 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Make                              7385 non-null   object 
 1   Model                             7385 non-null   object 
 2   Vehicle Class                     7385 non-null   object 
 3   Engine Size(L)                    7385 non-null   float64
 4   Cylinders                         7385 non-null   int64  
 5   Transmission                      7385 non-null   object 
 6   Fuel Type                         7385 non-null   object 
 7   Fuel Consumption City (L/100 km)  7385 non-null   float64
 8   Fuel Consumption Hwy (L/100 km)   7385 non-null   float64
 9   Fuel Consumption Comb (L/100 km)  7385 non-null   float64
 10  Fuel Consumption Comb (mpg)       7385 non-null   int64  
 11  CO2 Emissions(g/km)               7385 non-null   int64  
dtypes: flo

In [7]:
X = ds.drop("CO2 Emissions(g/km)", axis=1)
y = ds["CO2 Emissions(g/km)"]

In [8]:
X.shape

(7385, 11)

In [9]:
y.shape

(7385,)

In [10]:
numcols = ["Engine Size(L)","Cylinders","Fuel Consumption City (L/100 km)", "Fuel Consumption Hwy (L/100 km)", "Fuel Consumption Comb (L/100 km)", "Fuel Consumption Comb (mpg)"]
catcols = ["Make", "Model", "Vehicle Class", "Transmission", "Fuel Type"]

In [17]:
numpl = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale", StandardScaler())
])

catpl = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

tf = ColumnTransformer([
    ("numpl", numpl, numcols),
    ("catpl", catpl, catcols)
])

pl = Pipeline([
    ("transformer", tf),
    ("model", RandomForestRegressor())
])

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
pl.fit(X_train, y_train)

Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('numpl',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer()),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['Engine Size(L)',
                                                   'Cylinders',
                                                   'Fuel Consumption City '
                                                   '(L/100 km)',
                                                   'Fuel Consumption Hwy '
                                                   '(L/100 km)',
                                                   'Fuel Consumption Comb '
                                                   '(L/100 km)',
                                                   'Fuel Consumption Comb '
                                                   '(mpg)']),
                                                 ('catpl',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Make', 'Model',
                                                   'Vehicle Class',
                                                   'Transmission',
                                                   'Fuel Type'])])),
                ('model', RandomForestRegressor())])

In [20]:
y_pred = pl.predict(X_test)

In [21]:
y_pred

array([253.03      , 342.90414286, 326.98      , ..., 182.35      ,
       238.31666667, 191.58333333])

In [22]:
y_test

7261    253
4489    344
1539    322
3532    297
6418    308
       ... 
7185    221
1740    260
4609    184
1844    235
5154    193
Name: CO2 Emissions(g/km), Length: 1477, dtype: int64

In [23]:
mse = mean_squared_error(y_test, y_pred)
mse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2s = r2_score(y_test, y_pred)

print(f"mse:{mse:.2f}, mae:{mae:.2f}, r2s: {r2s:.2f}")

mse:3.50, mae:1.84, r2s: 1.00


In [24]:
joblib.dump(pl,"co2ecpl.joblib")

['co2ecpl.joblib']

In [ ]:
lpl = joblib.load("co2ecpl.joblib")

In [ ]:
lpl.predict(X_test)

In [ ]:
X_test